In [3]:
import pandas as pd
import requests
import io

In [4]:
# Helper to convert Bengali numerals to English
def convert_bn_numerals(text):
    bn_to_en = {'০':'0', '১':'1', '২':'2', '৩':'3', '৪':'4', '৫':'5', '৬':'6', '৭':'7', '৮':'8', '৯':'9'}
    for bn, en in bn_to_en.items():
        text = text.replace(bn, en)
    return text

In [ ]:


all_pages = []
base_url = "https://erp.powergrid.gov.bd/w/generations/view_generations_bn?page="
page_no = 6

for page in range(1, page_no):
    url = f"{base_url}{page}"
    response = requests.get(url, verify=False)
    
    if response.status_code == 200:
        # Use pandas to read the HTML table directly
        tables = pd.read_html(io.StringIO(response.text))
        if tables:
            all_pages.append(tables[0])
            print(f"Page {page} extracted.")

# Combine all pages into one DataFrame
if all_pages:
    # 1. Combine all pages
    final_df = pd.concat(all_pages, ignore_index=True)
    
    # 2. FIX: Flatten MultiIndex columns if they exist
    if isinstance(final_df.columns, pd.MultiIndex):
        # Joins the levels with an underscore, e.g., ('ভারত', 'ত্রিপুরা') -> 'ভারত_ত্রিপুরা'
        final_df.columns = [
            '_'.join([str(c) for c in col if "Unnamed" not in str(c)]).strip() 
            for col in final_df.columns.values
        ]

    # 3. Convert all numerals to English
    # Note: Use .map() for pandas 2.1.0+, or .applymap() for older versions
    final_df = final_df.map(lambda x: convert_bn_numerals(str(x)) if pd.notnull(x) else x)
    


    

c:\Users\zoop\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'erp.powergrid.gov.bd'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Page 1 extracted.
Page 2 extracted.


c:\Users\zoop\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'erp.powergrid.gov.bd'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\zoop\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'erp.powergrid.gov.bd'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Page 3 extracted.
Page 4 extracted.


c:\Users\zoop\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'erp.powergrid.gov.bd'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\zoop\AppData\Local\Programs\Python\Python314\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'erp.powergrid.gov.bd'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Page 5 extracted.


In [21]:
column_map = {
    'তারিখ_তারিখ': 'Date',
    'সময়_সময়': 'Time',
    'উৎপাদন(মেঃওঃ)_উৎপাদন(মেঃওঃ)': 'Generation_MW',
    'চাহিদা(মেঃওঃ)_চাহিদা(মেঃওঃ)': 'Demand_MW',
    'লোডশেড_লোডশেড': 'Loadshed',
    'গ্যাস_গ্যাস': 'Gas',
    'তরল জ্বালানী_তরল জ্বালানী': 'LPG',
    'কয়লা_কয়লা': 'Coal',
    'হাইড্রো_হাইড্রো': 'Hydro',
    'সৌর_সৌর': 'Solar',
    'বায়ু_বায়ু': 'Wind',
    'ভারত_ভেড়ামারা এইচভিডিসি': 'India_Bheramara',
    'ভারত_ত্রিপুরা': 'India_Tripura',
    'ভারত_আদানি': 'India_Adani',
    'নেপাল_নেপাল': 'Nepal',
    'মন্তব্য_মন্তব্য': 'Comment'
    }
final_df.rename(columns=column_map, inplace=True)
    # 4. Now save to Excel with index=False
final_df.to_excel("PowerGrid_Data_Fixed.xlsx", index=False)
print("Success! MultiIndex flattened and data saved.")

Success! MultiIndex flattened and data saved.


In [22]:
final_df.columns

Index(['Date', 'Time', 'Generation_MW', 'Demand_MW', 'Loadshed', 'Gas', 'LPG',
       'Coal', 'Hydro', 'Solar', 'Wind', 'India_Bheramara', 'India_Tripura',
       'India_Adani', 'Nepal', 'Comment'],
      dtype='str')